In [20]:
import json
from io import BytesIO
import os
from datetime import datetime, timezone
from pathlib import Path
import boto3
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASS_NAMES = ["Small", "Medium", "Large"]
NORMALIZATION = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}


from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env")


True

In [27]:
#local training settings and s3 paths
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 0.001
AWS_PROFILE = os.environ.get("AWS_PROFILE", "ATLAS")
AWS_REGION = "us-east-2"
BUCKET = "atlas-acm-research-f26"
IMAGE_PREFIX = "satellite/processed/"
METADATA_KEY = "dataset/im3_atlas_cleaned.csv"
OUTPUT_PREFIX = "artifacts/baseline/"

# All data and results stay in memory until results are uploaded to S3.
job_name = datetime.now(timezone.utc).strftime("local-%Y%m%d-%H%M%S-%f")
s3_output = f"{OUTPUT_PREFIX.rstrip('/')}/{job_name}/"

torch.manual_seed(42)
import sys
print(f"Python: {sys.executable}")
if not torch.cuda.is_available():
    raise RuntimeError(
       "This requires CUDA to use GPU power!!!"
    )
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")
if device.type == "cuda":
    print(f"Training GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA unavailable: using CPU.")
# Use the named profile when available, otherwise the standard AWS credential chain.
available_profiles = boto3.Session().available_profiles
aws_session = boto3.Session(
    profile_name=AWS_PROFILE if AWS_PROFILE in available_profiles else None,
    region_name=AWS_REGION,
)
if aws_session.get_credentials() is None:
    raise RuntimeError(
        "AWS credentials are missing. Configure the ATLAS profile with "
        "`aws configure --profile ATLAS`" 
    )
s3 = aws_session.client("s3")
print(f"Run: {job_name} | Device: {device}")


Python: c:\Users\kevin\Projects\ATLAS\.venv\Scripts\python.exe
PyTorch: 2.11.0+cu128 | CUDA build: 12.8
Training GPU: NVIDIA GeForce RTX 5070
Run: local-20261024-050751-428092 | Device: cuda


In [22]:
#read CSV into RAM and list tensor keys
def read_s3_bytes(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    try:
        return body.read()
    finally:
        body.close()

metadata = pd.read_csv(BytesIO(read_s3_bytes(METADATA_KEY)), dtype={"id": str})
tensor_keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=IMAGE_PREFIX):
    tensor_keys.extend(item["Key"] for item in page.get("Contents", []) if item["Key"].endswith(".pt"))
if not tensor_keys:
    raise ValueError(f"No tensors found at s3://{BUCKET}/{IMAGE_PREFIX}")
tensor_cache = {}  # Shared CPU RAM cache avoids fetching the same tensor each epoch.
print(f"Found {len(tensor_keys)} tensors in S3; metadata loaded into memory.")


Found 250 tensors in S3; metadata loaded into memory.


In [23]:
#baseline dataset reads tensors from S3 into RAM
class DataCenterDataset(Dataset):
    def __init__(self, frame):
        self.rows = list(frame[["s3_key", "size_label"]].itertuples(index=False, name=None))
        self.normalize = transforms.Normalize(**NORMALIZATION)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        key, label = self.rows[index]
        if key not in tensor_cache:
            data = torch.load(BytesIO(read_s3_bytes(key)), map_location="cpu", weights_only=True)
            tensor_cache[key] = data["image"].float()
        return self.normalize(tensor_cache[key]), int(label)

def load_and_split_data(tensor_keys, metadata):
    metadata = metadata.copy()

    metadata["sqft"] = pd.to_numeric(metadata["sqft"], errors="coerce")
    metadata = metadata[metadata["sqft"] > 0].copy()

    q1, q3 = metadata["sqft"].quantile([0.25, 0.75])
    metadata["size_label"] = metadata["sqft"].apply(
        lambda x: 0 if x < q1 else 1 if x <= q3 else 2
    )

    records = [
        {"id": Path(key).stem, "s3_key": key}
        for key in tensor_keys
    ]

    images = pd.DataFrame(records).merge(
        metadata[["id", "size_label"]], on="id"
    )

    facilities = images[["id", "size_label"]].drop_duplicates()

    train, remainder = train_test_split(
        facilities,
        test_size=0.30,
        random_state=42,
        stratify=facilities["size_label"]
    )

    validation, test = train_test_split(
        remainder,
        test_size=0.50,
        random_state=42,
        stratify=remainder["size_label"]
    )

    frames = [
        images[images["id"].isin(split["id"])].copy()
        for split in (train, validation, test)
    ]

    return frames, {"q1": float(q1), "q3": float(q3)}

frames, thresholds = load_and_split_data(tensor_keys, metadata)
loaders = [
    DataLoader(DataCenterDataset(frame), batch_size=BATCH_SIZE, shuffle=index == 0, num_workers=0)
    for index, frame in enumerate(frames)
]
for name, frame in zip(("train", "validation", "test"), frames):
    print(f"{name}: {len(frame)} tensors, {frame['id'].nunique()} facilities")
print("Square-footage thresholds:", thresholds)


train: 161 tensors, 161 facilities
validation: 35 tensors, 35 facilities
test: 35 tensors, 35 facilities
Square-footage thresholds: {'q1': 58043.5, 'q3': 231044.0}


In [24]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    model.eval()
    if optimizer is not None:
        model.fc.train()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            #clear the previous batch's gradients before calculating new ones.
            if optimizer is not None:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            #calculate gradients and use them to update the trainable weights.
            if optimizer is not None:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

#reuse pretrained image features and train a new three-class classification head.
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.to(device)

#cross-entropy measures classification error; Adam updates only the new head.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
history = []
print(f"Job: {job_name} | Device: {device}", flush=True)

#learn from the training set, then check validation performance after every epoch.
for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(model, loaders[0], criterion, device, optimizer)
    val_loss, val_accuracy = run_epoch(model, loaders[1], criterion, device)
    history.append(dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_accuracy,
                        val_loss=val_loss, val_accuracy=val_accuracy))
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_accuracy:.2%} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2%}", flush=True)



Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to C:\Users\kevin/.cache\torch\hub\checkpoints\resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:15<00:00, 5.50MB/s]


Job: local-20261024-045241-671400 | Device: cuda
Epoch 1/100 | Train Loss: 1.0572 | Train Acc: 50.31% | Val Loss: 1.0483 | Val Acc: 54.29%
Epoch 2/100 | Train Loss: 0.9666 | Train Acc: 56.52% | Val Loss: 1.0200 | Val Acc: 37.14%
Epoch 3/100 | Train Loss: 0.9237 | Train Acc: 55.90% | Val Loss: 0.9817 | Val Acc: 54.29%
Epoch 4/100 | Train Loss: 0.8700 | Train Acc: 59.63% | Val Loss: 0.9612 | Val Acc: 40.00%
Epoch 5/100 | Train Loss: 0.8397 | Train Acc: 59.01% | Val Loss: 0.9666 | Val Acc: 48.57%
Epoch 6/100 | Train Loss: 0.8514 | Train Acc: 57.76% | Val Loss: 0.9401 | Val Acc: 51.43%
Epoch 7/100 | Train Loss: 0.8072 | Train Acc: 61.49% | Val Loss: 0.9246 | Val Acc: 54.29%
Epoch 8/100 | Train Loss: 0.7732 | Train Acc: 62.73% | Val Loss: 0.9110 | Val Acc: 54.29%
Epoch 9/100 | Train Loss: 0.7561 | Train Acc: 63.35% | Val Loss: 0.9052 | Val Acc: 57.14%
Epoch 10/100 | Train Loss: 0.7534 | Train Acc: 62.73% | Val Loss: 0.9185 | Val Acc: 51.43%
Epoch 11/100 | Train Loss: 0.8054 | Train Acc: 62.

In [ ]:
#produce metrics and plot in RAM
def figure_bytes(figure):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=150)
    return buffer.getvalue()

def save_evaluation(model, loader, device, history, results):
    predictions, actual = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            predictions.extend(model(images.to(device)).argmax(1).cpu().tolist())
            actual.extend(labels.tolist())
    #rows are actual classes, columns are predicted classes.
    matrix = confusion_matrix(actual, predictions, labels=[0, 1, 2])
    report_args = dict(labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0)
    report = classification_report(actual, predictions, **report_args)
    metrics = {
        "test_accuracy": float(accuracy_score(actual, predictions)),
        "test_macro_f1": float(
            f1_score(
                actual,
                predictions,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "classification_report": classification_report(actual, predictions, output_dict=True, **report_args),
        "confusion_matrix": matrix.tolist(),
    }
    results["metrics.json"] = json.dumps(metrics, indent=2).encode("utf-8")
    results["classification_report.txt"] = report.encode("utf-8")
    results["training_history.csv"] = pd.DataFrame(history).to_csv(index=False).encode("utf-8")
    print(f"Test accuracy: {metrics['test_accuracy']:.4f} | Macro F1: {metrics['test_macro_f1']:.4f}")
    print(report, flush=True)

    #compare training and validation over time to show learning and possible overfitting.
    for metric in ("loss", "accuracy"):
        fig, ax = plt.subplots()
        for split, label in (("train", "Training"), ("val", "Validation")):
            ax.plot([row["epoch"] for row in history],
                    [row[f"{split}_{metric}"] for row in history], label=label)
        ax.set(xlabel="Epoch", ylabel=metric.capitalize(),
               title=f"Training and validation {metric}")
        if metric == "accuracy":
            ax.set_ylim(0, 1)
        ax.legend()
        fig.tight_layout()
        results[f"{metric}_curve.png"] = figure_bytes(fig)
        plt.close(fig)
    display = ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES)
    display.plot(cmap="Blues", values_format="d")
    display.ax_.set_title("Test confusion matrix")
    display.figure_.tight_layout()
    results["confusion_matrix.png"] = figure_bytes(display.figure_)
    plt.close(display.figure_)

results = {}
save_evaluation(model, loaders[2], device, history, results)
results["data_splits.csv"] = pd.concat([
    frame.assign(split=name)
    for name, frame in zip(("train", "validation", "test"), frames)
]).to_csv(index=False).encode("utf-8")
#save trained weights in RAM, the upload cell sends this under the current job.
buffer = BytesIO()
torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()}, buffer)
results["model.pkl"] = buffer.getvalue()
print("Results prepared in memory:", ", ".join(sorted(results)))


Test accuracy: 0.6000 | Macro F1: 0.5407
              precision    recall  f1-score   support

       Small       0.50      0.33      0.40         6
      Medium       0.76      0.68      0.72        19
       Large       0.43      0.60      0.50        10

    accuracy                           0.60        35
   macro avg       0.56      0.54      0.54        35
weighted avg       0.62      0.60      0.60        35

Results prepared in memory: accuracy_curve.png, classification_report.txt, confusion_matrix.png, data_splits.csv, loss_curve.png, metrics.json, training_history.csv


In [26]:
#upload to s3 from RAM
if not results:
    raise RuntimeError("Run evaluation before uploading results.")
if "model.pkl" not in results:
    raise RuntimeError("Run the evaluation cell to prepare model.pkl before uploading.")
content_types = {".png": "image/png", ".json": "application/json",
                 ".csv": "text/csv", ".txt": "text/plain", ".pkl": "application/octet-stream"}
for name, payload in sorted(results.items()):
    s3.upload_fileobj(BytesIO(payload), BUCKET, f"{s3_output}{name}",
                      ExtraArgs={"ContentType": content_types[Path(name).suffix]})
    print(f"Uploaded: {name}")
print(f"Metrics and visuals: s3://{BUCKET}/{s3_output}")


Uploaded: accuracy_curve.png
Uploaded: classification_report.txt
Uploaded: confusion_matrix.png
Uploaded: data_splits.csv
Uploaded: loss_curve.png
Uploaded: metrics.json
Uploaded: training_history.csv
Metrics and visuals: s3://atlas-acm-research-f26/artifacts/baseline/local-20261024-045241-671400/
